In [6]:
import trafilatura
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
import tempfile
import requests
import json
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Configuration
KEYWORDS = ["licenta", "master", "doctorat", "calendar", "taxe", "acte", "admitere", "candidati"]
BAD_KEYWORDS = ["anunt", "finalizare", "xls", "teze", "docx", "plan", "pdf", "uploads", "download", "2000", "2001", "2002", "2003", "2004", "2005", "2006", "2007", "2008", "2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]
START_URLS = ["https://www.tuiasi.ro/admitere", 
              "https://ac.tuiasi.ro/admitere", 
              "https://arh.tuiasi.ro/admitere-2025/", 
              "https://etti.tuiasi.ro/admitere", 
              "https://icpm.tuiasi.ro/admitere/", 
              "https://ci.tuiasi.ro/admitere/", 
              "https://cmmi.tuiasi.ro/admitere/", 
              "https://hgim.tuiasi.ro/admitere/",  
              "https://ieeia.tuiasi.ro/admitere/", 
              "https://mec.tuiasi.ro/admitere/", 
              "https://sim.tuiasi.ro/admitere/", 
              "https://dima.tuiasi.ro/admitere/"]    


In [4]:
import re


def _url_path_tokens(parsed_url):
    # Tokenize only the PATH so we don't match inside domains like "practeh".
    # ".../acte-de-studii" -> ["acte", "de", "studii"]
    return [t for t in re.split(r"[^a-z0-9]+", parsed_url.path.lower()) if t]


def get_page_links(url):
    """
    Extracts and cleans internal links from a page, filtering only those
    that contain specific admission-related keywords as WHOLE tokens in the URL path.

    Example: ".../proiecte/practeh" should NOT match keyword "acte".
    """
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return []

        soup = BeautifulSoup(downloaded, "html.parser")
        absolute_links = set()
        base_domain = urlparse(url).netloc

        for a_tag in soup.find_all("a", href=True):
            href = a_tag["href"].lower()
            full_url = urljoin(url, href)

            parsed_url = urlparse(full_url)
            clean_link = f"{parsed_url.scheme}://{parsed_url.netloc}{parsed_url.path}".rstrip("/")

            # 1. Domain Check: Must stay on the same faculty/university domain
            if base_domain in parsed_url.netloc and parsed_url.scheme in ["http", "https"]:
                tokens = _url_path_tokens(parsed_url)

                # 2. Keyword Check (whole-token): only match complete tokens in the PATH
                has_good_word = any(word in tokens for word in KEYWORDS)
                has_bad_word = any(word in tokens for word in BAD_KEYWORDS)

                if has_good_word and not has_bad_word:
                    absolute_links.add(clean_link)

        return list(absolute_links)
    except Exception as e:
        print(f"⚠️ Error fetching links from {url}: {e}")
        return []

In [7]:
queue = [(url, 0) for url in START_URLS]
visited = set()
all_discovered_links = set()
MAX_DEPTH = 5

while queue:
    current_url, depth = queue.pop(0)
    
    if current_url in visited:
        continue
    
    visited.add(current_url)
    all_discovered_links.add(current_url)
    
    prefix = "  " * depth + "└─"
    print(f"{prefix} [{depth}] Visiting: {current_url}")
    
    if depth < MAX_DEPTH:
        sublinks = get_page_links(current_url)
        
        new_links_found = 0
        for link in sublinks:
            if link not in visited:
                queue.append((link, depth + 1))
                new_links_found += 1
        
        if new_links_found > 0:
            pass 

with open("urls.txt", "w", encoding="utf-8") as f:
    for link in sorted(all_discovered_links):
        f.write(f"{link}\n")

print("💾 List saved in 'urls.txt'.")

└─ [0] Visiting: https://www.tuiasi.ro/admitere
└─ [0] Visiting: https://ac.tuiasi.ro/admitere
└─ [0] Visiting: https://arh.tuiasi.ro/admitere-2025/
└─ [0] Visiting: https://etti.tuiasi.ro/admitere
└─ [0] Visiting: https://icpm.tuiasi.ro/admitere/
└─ [0] Visiting: https://ci.tuiasi.ro/admitere/
└─ [0] Visiting: https://cmmi.tuiasi.ro/admitere/
└─ [0] Visiting: https://hgim.tuiasi.ro/admitere/
└─ [0] Visiting: https://ieeia.tuiasi.ro/admitere/
└─ [0] Visiting: https://mec.tuiasi.ro/admitere/
└─ [0] Visiting: https://sim.tuiasi.ro/admitere/
└─ [0] Visiting: https://dima.tuiasi.ro/admitere/
  └─ [1] Visiting: https://www.tuiasi.ro/admitere/doctorat
  └─ [1] Visiting: https://www.tuiasi.ro/doctorat
  └─ [1] Visiting: https://www.tuiasi.ro/admitere/master
  └─ [1] Visiting: https://www.tuiasi.ro/licenta/metode-de-selectie
  └─ [1] Visiting: https://www.tuiasi.ro/acte-de-studii
  └─ [1] Visiting: https://www.tuiasi.ro/licenta/dosarul-de-inscriere
  └─ [1] Visiting: https://www.tuiasi.ro/doct

In [8]:
def get_page_content(url):
    """
    Extracts only the clean text from a page.
    Best for feeding the LLM or building the vector DB.
    """
    try:
        if url.lower().endswith(".pdf"):
            loader = PyPDFLoader(url)
            pages = loader.load()
            return "\n".join([p.page_content for p in pages])
        else:
            downloaded = trafilatura.fetch_url(url)
            return trafilatura.extract(downloaded) if downloaded else None
    except Exception as e:
        print(f"⚠️ Error fetching content from {url}: {e}")
        return None

def get_faculty_tag(url):
    domain = urlparse(url).netloc
    parts = domain.split('.')
    
    faculty_map = {
        "ac": "Automatică și Calculatoare",
        "arh": "Arhitectură",
        "icpm": "Inginerie Chimică și Protecția Mediului",
        "ci": "Construcții și Instalații",
        "dima": "Design Industrial și Managementul Afacerilor",
        "ieeia": "Inginerie Electrică, Energetică și Informatică Aplicată",
        "etti": "Electronică, Telecomunicații și Tehnologii Informaționale",
        "hgim": "Hidrotehnică, Geodezie și Ingineria Mediului",
        "mec": "Mecanică",
        "sim": "Știința și Ingineria Materialelor",
        "cmmi": "Construcții de Mașini și Management Industrial"
    }

    if len(parts) >= 3 and parts[-2] == "tuiasi":
        abbr = parts[0].lower()
        if abbr in faculty_map:
            return f"{faculty_map[abbr]} ({abbr.upper()})"
        else:
            return "TUIASI" 
            
    return "TUIASI"

def get_url_tags(url):
    url_lower = url.lower()
    base_keywords = ["licenta", "master", "doctorat", "candidati", "taxe", "studii", "studies"]
    found_tags = [word for word in base_keywords if word in url_lower]
    if "/en/" in url_lower:
        found_tags.append("english")    
    return found_tags

def get_all_metadata_tags(url):
    faculty = get_faculty_tag(url)
    category_tags = get_url_tags(url)
    return [faculty] + category_tags

print(get_all_metadata_tags("https://ac.tuiasi.ro/en/students/didactic/calendar"))

['Automatică și Calculatoare (AC)', 'english']


In [9]:
OUTPUT_FILE = "database_documents.json"
CHUNK_SIZE = 1300
CHUNK_OVERLAP = int(1300 * 0.15)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

if not os.path.exists("urls.txt"):
    print("❌ Error: 'urls.txt' not found.")
    exit()

with open("urls.txt", "r", encoding="utf-8") as f:
    urls = [line.strip() for line in f.readlines() if line.strip()]

print(f"🚀 Starting processing for {len(urls)} URLs...")
print(f"⚙️ Chunk Settings: {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap.")

for url in urls:
    try:
        raw_text = get_page_content(url)

        if not raw_text or len(raw_text.strip()) < 50:
            print(f"⚠️ Ignored (Content too short or empty): {url}")
            continue

        tags = get_all_metadata_tags(url)
        # Put source URL next to tags in the text header for transparency/debugging.
        tag_header = f"{', '.join(tags)}\nSOURCE: {url}\n\n"

        chunks = text_splitter.split_text(raw_text)

        with open(OUTPUT_FILE, "a", encoding="utf-8") as file_out:
            for chunk in chunks:
                final_content = tag_header + chunk

                json_obj = {
                    "page_content": final_content,
                    "source": url,
                    "tags": tags,
                }
                file_out.write(json.dumps(json_obj, ensure_ascii=False) + "\n")

        print(f"   ✅ Processed and saved {len(chunks)} chunks for: {url}")

    except Exception as e:
        print(f"⚠️ Error processing {url}: {e}")

print(f"\n✨ Done! All plain documents have been successfully saved to '{OUTPUT_FILE}'.")

🚀 Starting processing for 164 URLs...
⚙️ Chunk Settings: 1300 chars, 195 overlap.
⚠️ Ignored (Content too short or empty): http://www.ci.tuiasi.ro/ro/admitere/admitere-studii-licenta/studii-universitare-de-licenta-admitere
⚠️ Ignored (Content too short or empty): http://www.ci.tuiasi.ro/ro/admitere/admitere-studii-master/studii-universitare-de-master-admitere
   ✅ Processed and saved 1 chunks for: https://ac.tuiasi.ro/admitere
   ✅ Processed and saved 2 chunks for: https://ac.tuiasi.ro/admitere/continuare-studii
   ✅ Processed and saved 16 chunks for: https://ac.tuiasi.ro/admitere/doctorat
   ✅ Processed and saved 14 chunks for: https://ac.tuiasi.ro/admitere/licenta
   ✅ Processed and saved 9 chunks for: https://ac.tuiasi.ro/admitere/masterat
   ✅ Processed and saved 3 chunks for: https://ac.tuiasi.ro/admitere/studii-postuniversitare
   ✅ Processed and saved 1 chunks for: https://ac.tuiasi.ro/en/students/didactic/calendar
   ✅ Processed and saved 2 chunks for: https://ac.tuiasi.ro/en/s

In [11]:
INPUT_FILE = "database_documents.json"
DB_DIR = "./chroma_db"

def load_documents_from_jsonl(file_path):
    """Reads documents from the previously generated JSONL file."""
    docs = []
    if not os.path.exists(file_path):
        print(f"❌ Error: File {file_path} not found.")
        return docs

    print(f"📂 Reading documents from {file_path}...")
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                docs.append(
                    Document(
                        page_content=data.get("page_content", ""),
                        metadata={
                            "tags": data.get("tags", []),
                            "source": data.get("source"),
                        },
                    )
                )

    return docs

def build_vector_database():
    """Processes chunks and stores them in the ChromaDB vector database."""
    documents = load_documents_from_jsonl(INPUT_FILE)
    
    if not documents:
        print("⚠️ No documents found to populate the database.")
        return

    print(f"✅ Loaded {len(documents)} fragments (chunks).")
    
    # Using a high-performance multilingual embedding model for Romanian/English
    print("🧠 Downloading/Loading embedding model (paraphrase-multilingual)...")
    embeddings_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

    print("🏗️ Building ChromaDB vector database. This might take a few moments...")
    
    # Initialize Chroma and save the data to the local directory
    vector_db = Chroma.from_documents(
        documents=documents,
        embedding=embeddings_model,
        persist_directory=DB_DIR
    )
    
    print(f"✨ Done! The database has been successfully saved to the '{DB_DIR}' directory.")
    return vector_db

db = build_vector_database()

📂 Reading documents from database_documents.json...
✅ Loaded 736 fragments (chunks).
🧠 Downloading/Loading embedding model (paraphrase-multilingual)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2507.90it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏗️ Building ChromaDB vector database. This might take a few moments...
✨ Done! The database has been successfully saved to the './chroma_db' directory.
